# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided workflow for loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library, referencing all Croissant entities using their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an mlcroissant DatasetMetadata object

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Publication date: {metadata.date_published}")
print(f"Keywords: {metadata.keywords}")


## 2. Data Overview
Review the available record sets, fields (columns), and entity `@id`s in the dataset.

**Note:** All entities are referenced by `@id`. This allows consistent programmatic referencing throughout the notebook.


In [ ]:
# List all record sets and their fields, referencing by @id

overview = []
record_set_ids = []
for rs in metadata.record_sets:
    print(f"-- RecordSet: {rs.name} (@id: {rs.id})")
    record_set_ids.append(rs.id)
    print("   Fields (columns):")
    for field in rs.fields:
        print(f"     - {field.name} (@id: {field.id}, dataType: {getattr(field, 'data_type', 'N/A')})")
    print()

# Store for later
record_set_ids = [rs.id for rs in metadata.record_sets]


## 3. Data Extraction
Load records from a selected record set into a Pandas DataFrame for analysis. Use the record set and field `@id`s identified above.

We'll use the first record set for demonstration. If the dataset contains multiple record sets, adapt as needed.

In [ ]:
## Extract data from each record set (referenced by @id)
# List all record set @ids
print("All record set @ids found:")
for idx, rid in enumerate(record_set_ids):
    print(f"  [{idx}] {rid}")

dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for record set @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print("  [No records loaded]")

# Pick the first non-empty record_set for demonstration
main_rs_id = None
for rs_id, df in dataframes.items():
    if len(df) > 0:
        main_rs_id = rs_id
        break
if main_rs_id is not None:
    print(f"\nMain record set for analysis: {main_rs_id}\n")
    print("Columns:", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No usable record sets with records were found.")


## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records by numeric thresholds, normalizing fields, and grouping data. All entity references use their `@id`.

In [ ]:
from numpy import number

# Use main_rs_id as the main table. Select numeric fields for demonstration.
if main_rs_id is not None:
    df = dataframes[main_rs_id]
    # Find likely numeric columns based on data type info or pandas detection
    # Use the @id of the first field with numeric dtype
    field_id_map = {field.name: field.id for field in [f for rs in metadata.record_sets if rs.id == main_rs_id for f in rs.fields]}

    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    # If not detected, try to convert some known numeric-sounding columns
    if not numeric_cols:
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except:
                pass
        numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

    if numeric_cols:
        numeric_field = numeric_cols[0]
        # Find @id for this field
        numeric_field_id = field_id_map.get(numeric_field, numeric_field)

        print(f"Using numeric field '{numeric_field}' (@id: {numeric_field_id}) for analysis.")
        threshold = df[numeric_field].mean()  # Use mean as threshold for demo
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where '{numeric_field}' (@id: {numeric_field_id}) > {threshold:.2f}: {len(filtered_df)} records")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by a categorical (non-numeric) field
        other_cols = [c for c in df.columns if c != numeric_field]
        group_field = None
        for col in other_cols:
            if df[col].dtype == 'object':
                group_field = col
                break
        if group_field is not None:
            group_field_id = field_id_map.get(group_field, group_field)
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped by '{group_field}' (@id: {group_field_id}):")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No data frame loaded for EDA.")


## 5. Visualization
Visualize distributions or relationships between fields in the dataset, using their `@id` for reference.

In [ ]:
import matplotlib.pyplot as plt

if main_rs_id is not None and numeric_cols:
    # Histogram
    df = dataframes[main_rs_id]
    plt.figure(figsize=(7,4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field} (@id: {numeric_field_id})")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group, if group_field exists
    if group_field is not None:
        df.boxplot(column=numeric_field, by=group_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric data or record set for visualization.")


## 6. Conclusion
This notebook demonstrated how to:

- Load and inspect a FAIR²-annotated dataset via Croissant schema using `mlcroissant`.
- Enumerate record sets, fields, and their `@id`s for consistent referencing throughout analysis.
- Extract data into DataFrames and perform basic EDA steps including filtering, normalization, grouping, and plotting.

**Note:** For advanced machine learning or domain-specific analysis, tailor operations by deeply exploring the schema via `mlcroissant` and utilizing the rich metadata for reproducible science.
